# Data Profiling — MUSAN Dataset
Proyecto C: Audio Source Separation via NMF
**Optimización Numérica | Iberoamericana León 2026**

---
Ejecutar localmente con Jupyter.  
Los outputs (PNG + CSV) se suben a GitHub en la carpeta `outputs/`.

# 1. Imports y Configuración

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from scipy import stats
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

# Configuración de estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# Rutas
DATA_ROOT   = r'C:\Users\frank\OneDrive\Documentos\Ibero Semestres\6tosemestre\Python\data\musan'
OUTPUT_DIR  = r'C:\Users\frank\OneDrive\Documentos\Ibero Semestres\6tosemestre\Python\OptimizacionM\nmf-audio-separation\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parámetros del proyecto
N_PER_CAT   = 30 
N_FFT       = 1024
HOP_LENGTH  = 512
SR_TARGET   = 22050
CATEGORIES  = ['music', 'noise', 'speech']

# Paleta de colores
PALETTE = {
    'music': '#4C72B0', 'noise': '#DD8452', 'speech': '#55A868',
    'bg': '#F8F9FA', 'grid': '#DEE2E6', 'text': '#212529',
}
CAT_LABELS = {'music': 'Music', 'noise': 'Noise', 'speech': 'Speech'}

print(f'Datos  : {DATA_ROOT}')
print(f'Outputs: {OUTPUT_DIR}\n')
print('Setup completado. Librerías y rutas listas.')

# 2. EDA MUSAN

## Celda 1: Funciones Auxiliares

In [ ]:
# FUNCIONES DE PROCESAMIENTO

def scan_files(data_root, categories, n_per_cat=None, exts=('.wav','.flac','.mp3')):
    """Escanea carpetas y devuelve DataFrame con metadatos de archivos."""
    rows = []
    for cat in categories:
        path = Path(data_root) / cat
        if not path.exists(): continue
        files = [f for f in path.rglob('*') if f.suffix.lower() in exts]
        if n_per_cat and len(files) > n_per_cat:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(files), n_per_cat, replace=False)
            files = [files[i] for i in sorted(idx)]
        for f in files:
            rows.append({
                'category': cat, 'path': str(f), 'filename': f.name,
                'size_bytes': f.stat().st_size, 'subdir': str(f.parent.relative_to(path)),
            })
    return pd.DataFrame(rows)

def analyze_file(path, sr_target=SR_TARGET, max_dur=30.0):
    """Extrae características acústicas y de espectrograma de un archivo."""
    try:
        info = sf.info(path)
        y, sr = librosa.load(path, sr=sr_target, mono=True, duration=max_dur)
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        return {
            'ok': True, 'sr_original': info.samplerate, 'channels': info.channels,
            'duration_s': info.duration, 'format': info.format,
            'rms_energy': float(np.sqrt(np.mean(y**2))),
            'zero_cross_rate': float(np.mean(librosa.feature.zero_crossing_rate(y))),
            'spectral_centroid_hz': float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))),
            'spectral_bandwidth': float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))),
            'spectral_rolloff_hz': float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))),
            'stft_F': S.shape[0], 'stft_T': S.shape[1],
            'stft_mean': float(S.mean()), 'stft_max': float(S.max()),
        }
    except Exception as e:
        return {'ok': False, 'error': str(e)}

def ax_style(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor(PALETTE['bg'])
    ax.set_title(title, fontsize=9.5, fontweight='bold', pad=6, color=PALETTE['text'])
    ax.set_xlabel(xlabel, fontsize=8, color='#495057')
    ax.set_ylabel(ylabel, fontsize=8, color='#495057')
    ax.tick_params(labelsize=7.5, colors='#495057')
    ax.grid(True, color=PALETTE['grid'], linewidth=0.5, alpha=0.8, zorder=0)
    for sp in ax.spines.values(): sp.set_edgecolor(PALETTE['grid'])

## Celda 2: Extracción y Generación de Metadatos

In [ ]:
# EXTRACCIÓN DE DATOS Y METADATOS

print(f'Escaneando (muestra: {N_PER_CAT} por categoría)...')
df_files = scan_files(DATA_ROOT, CATEGORIES, n_per_cat=N_PER_CAT)

records = []
for _, row in tqdm(df_files.iterrows(), total=len(df_files), desc='Analizando'):
    meta = analyze_file(row['path'])
    meta.update(row.to_dict())
    records.append(meta)

df = pd.DataFrame(records)
df_ok = df[df['ok'] == True].copy()
errors = df[df['ok'] != True]

print(f'\n{len(df_ok)}/{len(df_files)} archivos OK')
if len(errors) > 0:
    print(f'{len(errors)} errores:')
    print(errors[['filename','error']].to_string(index=False))

# Guardar CSVs
df_ok.to_csv(f'{OUTPUT_DIR}/metadata_MUSAN.csv', index=False)
summary = df_ok.groupby('category').agg(
    n_archivos          = ('filename', 'count'),
    dur_total_min       = ('duration_s', lambda x: round(x.sum()/60, 2)),
    dur_media_s         = ('duration_s', lambda x: round(x.mean(), 2)),
    rms_media           = ('rms_energy', lambda x: round(x.mean(), 5)),
    zcr_media           = ('zero_cross_rate', lambda x: round(x.mean(), 5)),
    centroide_hz        = ('spectral_centroid_hz', lambda x: round(x.mean(), 1)),
    stft_F              = ('stft_F', 'first'),
    tamano_MB           = ('size_bytes', lambda x: round(x.sum()/1e6, 1)),
)
summary.to_csv(f'{OUTPUT_DIR}/summary_MUSAN.csv')

# Variables auxiliares para los gráficos
cats = [c for c in CATEGORIES if c in df_ok['category'].unique()]
colors = [PALETTE[c] for c in cats]
labels = [CAT_LABELS[c] for c in cats]
Categorias_limpias = ['Music', 'Noise', 'Speech']

display(summary)

## Celda 3: Análisis de Distribuciones Físicas (EDA Visual)

In [ ]:
# 4. VISUALIZACIÓN DE DISTRIBUCIONES

dur_tot = [df_ok[df_ok['category']==c]['duration_s'].sum()/60 for c in cats]

# --- FIGURA 1: Distribución de Duración (Pie y KDE) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=PALETTE['bg'])

# Pie Chart
wedges, texts, autotexts = ax1.pie(
    dur_tot, labels=None, colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2.5, alpha=0.9),
    textprops={'fontsize': 11, 'fontweight': 'bold'}, pctdistance=0.75,
    explode=[0.02] * len(cats)
)
for autotext in autotexts: autotext.set_color('white')
centre_circle = plt.Circle((0, 0), 0.60, fc='white', edgecolor=PALETTE['grid'], linewidth=1.5)
ax1.add_artist(centre_circle)
ax1.text(0, 0, f'Total\n{sum(dur_tot):.1f}\nmin', ha='center', va='center', fontsize=12, fontweight='bold')
ax1.legend(wedges, Categorias_limpias, loc='center left', bbox_to_anchor=(1, 0.5))
ax1.set_title('Duración Total', fontweight='bold')

# Histograma Duración
for cat, color, label in zip(cats, colors, labels):
    d = df_ok[df_ok['category']==cat]['duration_s'].dropna()
    ax2.hist(d, bins=35, color=color, alpha=0.6, label=label, edgecolor='white', lw=1.5, density=True)
    kde = stats.gaussian_kde(d)
    x_range = np.linspace(d.min(), d.max(), 200)
    ax2.plot(x_range, kde(x_range), color=color, lw=3, alpha=0.8)
    ax2.axvline(d.mean(), color=color, linestyle='--', lw=1.5, alpha=0.5)
ax2.set_title('Distribución de Duración de Audio', fontweight='bold')
ax2.set_xlabel('Segundos')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_distribucion_duracion.png", dpi=300)
plt.show()

# --- FIGURA 2: Distribución RMS Energy ---
fig3 = plt.figure(figsize=(13, 10), facecolor=PALETTE['bg'])
gs = gridspec.GridSpec(2, 2, figure=fig3, hspace=0.3, wspace=0.25)
rms_data = [df_ok[df_ok['category']==c]['rms_energy'].dropna().values for c in cats]

# KDE Superior
ax_top = fig3.add_subplot(gs[0, :])
for cat, color, label in zip(cats, colors, labels):
    d = df_ok[df_ok['category']==cat]['rms_energy'].dropna()
    ax_top.hist(d, bins=40, color=color, alpha=0.9, label=label, edgecolor='white', density=True)
    kde = stats.gaussian_kde(d)
    x_range = np.linspace(d.min(), d.max(), 200)
    ax_top.plot(x_range, kde(x_range), color=color, lw=2.5)
ax_top.legend(loc='upper right')
ax_top.set_title('RMS Energy Distribution', fontweight='bold')

# Boxplot Inferior Izq
ax_bl = fig3.add_subplot(gs[1, 0])
bp = ax_bl.boxplot(rms_data, patch_artist=True, widths=0.6,
                 medianprops=dict(color='white', linewidth=2.5))
for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color); patch.set_alpha(0.8)
ax_bl.set_xticklabels(labels)
ax_bl.set_title('Boxplot Comparativo RMS', fontweight='bold')

# Violin Plot Inferior Der
ax_br = fig3.add_subplot(gs[1, 1])
vp = ax_br.violinplot(rms_data, positions=range(len(cats)), showmeans=True, showmedians=True)
for pc, color in zip(vp['bodies'], colors): pc.set_facecolor(color); pc.set_alpha(0.7)
ax_br.set_xticks(range(len(cats))); ax_br.set_xticklabels(labels)
ax_br.set_title('Violin Plot RMS', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_energia_rms.png", dpi=300)
plt.show()

## Celda 4: Análisis de Señales Espectrales

In [ ]:
#  ANÁLISIS DE SEÑALES (ESPECTRO Y FORMA DE ONDA)

DUR_SHOW = 5.0
fig, axes = plt.subplots(3, 4, figsize=(22, 12), facecolor=PALETTE['bg'])
fig.suptitle('Análisis Acústico Completo: Waveform | Spectrogram | ZCR | X-Matrix Dist', fontweight='bold', y=1.02)

for row_idx, (cat, color, label) in enumerate(zip(cats, colors, labels)):
    path = df_ok[df_ok['category']==cat]['path'].iloc[0]
    y, sr = librosa.load(path, sr=SR_TARGET, mono=True, duration=DUR_SHOW)
    t = np.linspace(0, len(y)/sr, len(y))
    S = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    S_mag = np.abs(S)
    
    # 1. Forma de onda
    ax = axes[row_idx, 0]
    ax.plot(t, y, color=color, lw=0.8, alpha=0.9)
    ax.fill_between(t, y, 0, color=color, alpha=0.2)
    ax.set_title(f'{label} - Waveform')
    
    # 2. Espectrograma
    ax = axes[row_idx, 1]
    S_db = librosa.amplitude_to_db(S_mag, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='hz', cmap='magma', ax=ax)
    ax.set_title(f'Espectrograma')
    
    # 3. ZCR
    ax = axes[row_idx, 2]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    ax.plot(zcr, color=color, alpha=0.9)
    ax.set_title('Zero Crossing Rate')
    
    # 4. Distribución Matriz X (NMF target)
    ax = axes[row_idx, 3]
    ax.hist(S_mag.flatten(), bins=50, color=color, alpha=0.7, log=True)
    ax.set_title('Distribución STFT (Matriz X)')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_analisis_espectral.png", dpi=300)
plt.show()

## Celda 5: Reporte EDA

In [ ]:
# REPORTE FINAL Y PARÁMETROS NMF

print('  REPORTE — MUSAN Dataset  |  Proyecto NMF Audio Separation')
for cat in cats:
    sub = df_ok[df_ok['category']==cat]
    print(f'\n  {CAT_LABELS[cat]}')
    print(f'  {"─"*38}')
    print(f'  Archivos analizados    {len(sub):>8,}')
    print(f'  Duración total         {sub["duration_s"].sum()/60:>8.1f} min')
    print(f'  Sample rate común      {int(sub["sr_original"].mode()[0]):>8,} Hz')

print(f'\n  PARÁMETROS NMF DERIVADOS')
print(f'  {"─"*38}')
print(f'  N_FFT = {N_FFT}  →  F = {N_FFT//2+1} bins de frecuencia')
print(f'  Hop   = {HOP_LENGTH}')
print(f'  SR    = {SR_TARGET} Hz')
print(f'  Rango k = [5, 10, 15, 20]')

print(f'\n  ARCHIVOS GENERADOS  →  Listos para GitHub:')
print(f'  {"─"*38}')
for file in os.listdir(OUTPUT_DIR):
    print(f'  outputs/{file}')
print('=' * 62)

# 3. Fundamentos matemáticos (Sanity Checks) (W y H)

In [ ]:

# Configuración del caso miniatura (m=2, n=2, k=1)
X = np.array([[5.0, 3.0], 
              [2.0, 4.0]])

# W es de dimensiones (m, k) -> (2, 1)
W = np.array([[1.0], 
              [2.0]])

# H es de dimensiones (k, n) -> (1, 2)
H = np.array([[2.0, 1.0]])

# epsilon exigido por el proyecto
epsilon = 1e-5

# Función de pérdida base: f(W,H) = 0.5 * ||X - WH||_F^2
def f_loss(W, H, X):
    residual = (W @ H) - X
    return 0.5 * np.sum(residual ** 2)

# --- SANITY CHECK PARA W ---
grad_W_analitico = ((W @ H) - X) @ H.T
grad_W_numerico = np.zeros_like(W)
m, k = W.shape

for a in range(m):
    for b in range(k):
        E = np.zeros_like(W)
        E[a, b] = 1.0
        loss_plus = f_loss(W + epsilon * E, H, X)
        loss_base = f_loss(W, H, X)
        grad_W_numerico[a, b] = (loss_plus - loss_base) / epsilon

error_relativo_W = np.abs(grad_W_analitico - grad_W_numerico) / (np.abs(grad_W_analitico) + 1e-8)
error_maximo_W = np.max(error_relativo_W)

print("--- SANITY CHECK: GRADIENTES DE W ---")
print("Gradiente Analítico W:\n", grad_W_analitico)
print("\nGradiente Numérico W:\n", grad_W_numerico)
print(f"\nError relativo máximo W: {error_maximo_W:.2e}")
if error_maximo_W < 1e-4:
    print("✓ ÉXITO: El error relativo es menor a 1e-4.\n")
else:
    print("✗ FALLO: Revisa la derivación o la implementación.\n")


# --- SANITY CHECK PARA H ---
# grad_H = W^T(WH - X)
grad_H_analitico = W.T @ ((W @ H) - X)
grad_H_numerico = np.zeros_like(H)
k_dim, n_dim = H.shape

for c in range(k_dim):
    for d in range(n_dim):
        E_H = np.zeros_like(H)
        E_H[c, d] = 1.0
        loss_plus_H = f_loss(W, H + epsilon * E_H, X)
        loss_base_H = f_loss(W, H, X)
        grad_H_numerico[c, d] = (loss_plus_H - loss_base_H) / epsilon

error_relativo_H = np.abs(grad_H_analitico - grad_H_numerico) / (np.abs(grad_H_analitico) + 1e-8)
error_maximo_H = np.max(error_relativo_H)

print("--- SANITY CHECK: GRADIENTES DE H ---")
print("Gradiente Analítico H:\n", grad_H_analitico)
print("\nGradiente Numérico H:\n", grad_H_numerico)
print(f"\nError relativo máximo H: {error_maximo_H:.2e}")
if error_maximo_H < 1e-4:
    print("✓ ÉXITO: El error relativo de H es menor a 1e-4.\n")
else:
    print("✗ FALLO: Revisa la derivación o la implementación.\n")


# 2. IMPLEMENTACIÓN DE REGLAS (2.3 y 2.4)

def update_block(Block, Grad, v_Block, alpha, beta, method, is_nmf=True):
    """
    Aplica la variante de descenso de gradiente (2.3) y la proyección (2.4).
    """
    if method == 'gd':
        # 2.3.1 Vanilla GD
        Block = Block - alpha * Grad
        
    elif method == 'momentum':
        # 2.3.2 Momentum GD
        v_Block = beta * v_Block + Grad
        Block = Block - alpha * v_Block
        
    elif method == 'nesterov':
        # 2.3.3 Nesterov Accelerated Gradient
        # NOTA: En el algoritmo real, calculas un Grad_look usando Block_look. 
        # Aquí asumimos que 'Grad' ya es ese gradiente adelantado.
        v_Block = beta * v_Block + Grad
        Block = Block - alpha * v_Block

    # 2.4 Non-Negativity Constraints and Projections
    if is_nmf:
        # np.maximum proyecta todo valor negativo a 0
        Block = np.maximum(Block, 0)
        
    return Block, v_Block

# --- Mini prueba de las funciones ---
print("--- PRUEBA DE OPTIMIZACIÓN ---")
alpha_w = 0.01
beta = 0.9
v_W = np.zeros_like(W)

# Probando un paso de Vanilla GD con Proyección
W_new, v_W_new = update_block(W, grad_W_analitico, v_W, alpha_w, beta, method='gd', is_nmf=True)
print("W actualizado con Vanilla GD y Proyectado:\n", W_new)

# 4. Algoritmo 1 Unified Block-Coordinate Gradient Descent (BCGD)

In [ ]:

def unified_bcgd(X, k, steps=200, innerW=1, innerH=1, 
                 alpha_W=1e-3, alpha_H=1e-3, method='gd', beta=0.9, 
                 is_nmf=True, M=None, lmbda=0.0):
    """
    Algoritmo 1: Unified Block-Coordinate Gradient Descent for Matrix Factorization
    Configurado por defecto para Proyecto C (Audio NMF).
    """
    m, n = X.shape
    
    # Manejo de la máscara M (si no se manda, es una matriz de unos)
    if M is None:
        M = np.ones((m, n))
        
    # Función de proyección (Punto 2.4)
    def proj(matrix):
        if is_nmf:
            return np.maximum(matrix, 0)
        return matrix

    # 1. Inicialización (Valores positivos pequeños, recomendación Parte 5.1)
    # Usamos 1/sqrt(k) para controlar la escala inicial y evitar explosiones
    W = np.random.rand(m, k) / np.sqrt(k)
    H = np.random.rand(k, n) / np.sqrt(k)
    
    # 2. Inicialización de velocidades
    vW = np.zeros((m, k))
    vH = np.zeros((k, n))
    
    loss_history = []
    
    # 3. Bucle principal
    for s in range(steps):
        
        # ----------------------------------------------------
        # BLOQUE W (Líneas 4 a 20)
        # ----------------------------------------------------
        for t in range(innerW):
            if method == 'nesterov':
                # Paso adelantado (Lookahead)
                W_look = W - alpha_W * beta * vW
                R_look = M * ((W_look @ H) - X)
                gW_look = R_look @ H.T + lmbda * W_look
                
                # Actualización Nesterov
                vW = beta * vW + gW_look
                W = W - alpha_W * vW
                
            else:
                # Cálculo normal de residual y gradiente
                R = M * ((W @ H) - X)
                gW = R @ H.T + lmbda * W
                
                if method == 'gd':
                    W = W - alpha_W * gW
                elif method == 'momentum':
                    vW = beta * vW + gW
                    W = W - alpha_W * vW
            
            # Proyección (Punto 2.4)
            W = proj(W)
            
        # ----------------------------------------------------
        # BLOQUE H (Líneas 21 a 26)
        # ----------------------------------------------------
        for t in range(innerH):
            if method == 'nesterov':
                # Paso adelantado (Lookahead)
                H_look = H - alpha_H * beta * vH
                R_look = M * ((W @ H_look) - X)
                gH_look = W.T @ R_look + lmbda * H_look
                
                # Actualización Nesterov
                vH = beta * vH + gH_look
                H = H - alpha_H * vH
                
            else:
                # Cálculo normal de residual y gradiente
                R = M * ((W @ H) - X)
                gH = W.T @ R + lmbda * H
                
                if method == 'gd':
                    H = H - alpha_H * gH
                elif method == 'momentum':
                    vH = beta * vH + gH
                    H = H - alpha_H * vH
            
            # Proyección (Punto 2.4)
            H = proj(H)
            
        # ----------------------------------------------------
        # CÁLCULO DE LA PÉRDIDA (Línea 27)
        # ----------------------------------------------------
        R_final = M * ((W @ H) - X)
        loss = 0.5 * np.sum(R_final ** 2) + (lmbda / 2) * (np.sum(W ** 2) + np.sum(H ** 2))
        loss_history.append(loss)
        
        # Opcional: Imprimir progreso cada 50 iteraciones para no saturar la consola
        if (s + 1) % 50 == 0 or s == 0:
            print(f"Iteración {s + 1:4d}/{steps} | Loss: {loss:.4f}")
            
    return W, H, loss_history

# 5. Entrenamiento y Comparación

In [ ]:


print("--- PRUEBA RÁPIDA: MEZCLA DIGITAL Y ESPECTROGRAMAS ---")

# 1. Tu ruta real
DATA_ROOT = r"C:\Users\frank\OneDrive\Documentos\Ibero Semestres\6tosemestre\Python\data\musan"

# Buscar automáticamente un archivo de speech y uno de noise
ruta_speech = list(Path(DATA_ROOT).joinpath('speech').rglob('**/*.wav'))[0]
ruta_noise = list(Path(DATA_ROOT).joinpath('noise').rglob('**/*.wav'))[0]

print("1. Cargando y mezclando audios...")
print(f"   Voz: {ruta_speech.name}")
print(f"   Ruido: {ruta_noise.name}")

# Parámetros STFT
N_FFT = 1024
HOP_LENGTH = 512
SR_TARGET = 22050

# Cargar audios (usamos 15 segundos para esta prueba)
y_speech, _ = librosa.load(ruta_speech, sr=SR_TARGET, mono=True, duration=5.0)
y_noise, _  = librosa.load(ruta_noise, sr=SR_TARGET, mono=True, duration=5.0)

# Igualar tamaños y sumar (bajamos el ruido a la mitad para no tapar la voz)
min_len = min(len(y_speech), len(y_noise))
y_mix = y_speech[:min_len] + (0.5 * y_noise[:min_len])

# Sacar la Matriz X de la mezcla
S_mix = librosa.stft(y_mix, n_fft=N_FFT, hop_length=HOP_LENGTH)
X_real = np.abs(S_mix)

print(f"\n2. Dimensiones de la Matriz X (Mezcla): {X_real.shape}")

# ========================================================
# ENTRENAMIENTO DEL MODELO
# ========================================================
print("3. Entrenando modelo (Nesterov) para obtener W y H...")
k_model = 10  # Definimos el número de componentes
np.random.seed(42) # Fijamos semilla

# Entrenamos el modelo con la matriz mezclada
W_out, H_out, _ = unified_bcgd(
    X=X_real, k=k_model, steps=200, 
    innerW=1, innerH=1, 
    alpha_W=1e-3, alpha_H=1e-3, 
    method='nesterov', beta=0.9, 
    is_nmf=True, lmbda=0.0
)

# ========================================================
# GRÁFICAS (Requisito 10.2: Spectral bases y Temporal activations)
# ========================================================
print("\n4. Generando gráficas...")
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Descomposición NMF: Bases Espectrales (W) y Activaciones Temporales (H)', 
             fontsize=16, fontweight='bold')

# Graficar Matriz W (Bases Espectrales)
ax_W = axes[0]
W_db = librosa.amplitude_to_db(W_out, ref=np.max)
img_W = librosa.display.specshow(W_db, y_axis='hz', x_axis='frames', 
                                 sr=SR_TARGET, hop_length=HOP_LENGTH, 
                                 cmap='magma', ax=ax_W)
ax_W.set_title(f'Matriz W: {k_model} Bases Espectrales / Plantillas de Frecuencia', fontsize=12)
ax_W.set_xlabel('Componente (k)')
ax_W.set_ylabel('Frecuencia (Hz)')
fig.colorbar(img_W, ax=ax_W, format="%+2.0f dB")

# Graficar Matriz H (Activaciones Temporales)
ax_H = axes[1]
img_H = librosa.display.specshow(H_out, x_axis='time', y_axis='frames', 
                                 sr=SR_TARGET, hop_length=HOP_LENGTH, 
                                 cmap='viridis', ax=ax_H)
ax_H.set_title('Matriz H: Activaciones Temporales por Componente', fontsize=12)
ax_H.set_xlabel('Tiempo (s)')
ax_H.set_ylabel('Componente (k)')
fig.colorbar(img_H, ax=ax_H, label='Magnitud de Activación')

plt.tight_layout()
plt.show()

# 6. Evaluación Final (Splits y Algoritmo 2)

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("--- FASE FINAL: MEZCLA DIGITAL, SPLITS Y EVALUACIÓN ---")

# 1. Tu ruta real
DATA_ROOT = r"C:\Users\frank\OneDrive\Documentos\Ibero Semestres\6tosemestre\Python\data\musan"

# Buscar automáticamente un archivo de speech y uno de noise
# rglob('**/*.wav') busca en todas las subcarpetas
ruta_speech = list(Path(DATA_ROOT).joinpath('speech').rglob('**/*.wav'))[0]
ruta_noise = list(Path(DATA_ROOT).joinpath('noise').rglob('**/*.wav'))[0]

print(f"1. Mezclando audios:")
print(f"   Voz: {ruta_speech.name}")
print(f"   Ruido: {ruta_noise.name}")

# Parámetros STFT
N_FFT = 1024
HOP_LENGTH = 512
SR_TARGET = 22050

# Cargar audios (usamos 5 segundos para que la evaluación no tarde horas)
y_speech, _ = librosa.load(ruta_speech, sr=SR_TARGET, mono=True, duration=5.0)
y_noise, _  = librosa.load(ruta_noise, sr=SR_TARGET, mono=True, duration=5.0)

# Igualar tamaños y sumar (bajamos el ruido a la mitad para no tapar la voz)
min_len = min(len(y_speech), len(y_noise))
y_mix = y_speech[:min_len] + (0.5 * y_noise[:min_len])

# Sacar la Matriz X de la mezcla
S_mix = librosa.stft(y_mix, n_fft=N_FFT, hop_length=HOP_LENGTH)
X_real = np.abs(S_mix)

print(f"\n2. Dimensiones de la Matriz X (Mezcla): {X_real.shape}")

# ========================================================
# SPLIT CONTIGUO EN EL TIEMPO
# ========================================================
T = X_real.shape[1]
train_idx = int(0.70 * T)
val_idx = int(0.85 * T)

X_train = X_real[:, :train_idx]
X_val = X_real[:, train_idx:val_idx]
X_test = X_real[:, val_idx:]

print(f"   X_train (70%): {X_train.shape}")
print(f"   X_val   (15%): {X_val.shape}")
print(f"   X_test  (15%): {X_test.shape}\n")

# ========================================================
# ALGORITMO 2: EVALUACIÓN EN HELDOUT
# ========================================================
def eval_heldout(W_fixed, X_eval, innerH_eval=100, alpha_H=1e-3):
    m, k = W_fixed.shape
    n_eval = X_eval.shape[1]
    H_eval = np.random.rand(k, n_eval) / np.sqrt(k)
    
    for t in range(innerH_eval):
        g_H = W_fixed.T @ ((W_fixed @ H_eval) - X_eval)
        H_eval = np.maximum(H_eval - alpha_H * g_H, 0)
        
    rmse = np.sqrt(np.sum(((W_fixed @ H_eval) - X_eval)**2) / (m * n_eval))
    return rmse

# ========================================================
# PRUEBA DE RANGOS (k)
# ========================================================
k_values = [5, 10, 15, 20]
val_rmse_history = []

print("3. Evaluando diferentes valores de rango (k)...")
for k_test in k_values:
    print(f"   Entrenando W con k={k_test} en el set de Train...")
    np.random.seed(42)
    
    W_train, _, _ = unified_bcgd(
        X=X_train, k=k_test, steps=200, 
        method='nesterov', is_nmf=True
    )
    
    rmse_val = eval_heldout(W_train, X_val)
    val_rmse_history.append(rmse_val)
    print(f"   -> RMSE en Validación: {rmse_val:.4f}")

# Graficar Val RMSE vs k
plt.figure(figsize=(8, 5))
plt.plot(k_values, val_rmse_history, marker='o', color='#55A868', linewidth=2, markersize=8)
plt.title('Evaluación de Hiperparámetros: RMSE de Validación vs Rango (k)', fontsize=14, fontweight='bold')
plt.xlabel('Rango (k)', fontsize=12)
plt.ylabel('RMSE (Validación)', fontsize=12)
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# ========================================================
# EVALUACIÓN FINAL EN TEST
# ========================================================
best_k_idx = np.argmin(val_rmse_history)
best_k = k_values[best_k_idx]
print(f"\n4. El mejor k según validación fue k={best_k}. Evaluando en TEST...")

np.random.seed(42)
W_best, _, _ = unified_bcgd(X=X_train, k=best_k, steps=200, method='nesterov', is_nmf=True)

final_test_rmse = eval_heldout(W_best, X_test)
print(f"★ RMSE FINAL EN TEST: {final_test_rmse:.4f} ★")

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("--- FASE FINAL: MEZCLA DIGITAL CON LOOP, SPLITS Y EVALUACIÓN ---")

# 1. Tu ruta real
DATA_ROOT = r"C:\Users\frank\OneDrive\Documentos\Ibero Semestres\6tosemestre\Python\data\musan"

# Buscar automáticamente un archivo de speech y uno de noise
ruta_speech = list(Path(DATA_ROOT).joinpath('speech').rglob('**/*.wav'))[0]
ruta_noise = list(Path(DATA_ROOT).joinpath('noise').rglob('**/*.wav'))[0]

print(f"1. Cargando audios:")
print(f"   Voz: {ruta_speech.name}")
print(f"   Ruido: {ruta_noise.name}")

# Parámetros STFT
N_FFT = 1024
HOP_LENGTH = 512
SR_TARGET = 22050

# Cargar audios completos (sin límite de duración)
y_speech, _ = librosa.load(ruta_speech, sr=SR_TARGET, mono=True)
y_noise, _  = librosa.load(ruta_noise, sr=SR_TARGET, mono=True)

print(f"   Duración original voz: {len(y_speech)/SR_TARGET:.2f} s")
print(f"   Duración original ruido: {len(y_noise)/SR_TARGET:.2f} s")

# ========================================================
# EL TRUCO DEL BUCLE (LOOP) PARA EL RUIDO
# ========================================================
target_len = len(y_speech)

# Calculamos cuántas veces cabe el ruido en la voz (redondeando hacia arriba)
repeats = int(np.ceil(target_len / len(y_noise)))

# Repetimos el ruido (loop) y luego lo cortamos exactamente al tamaño de la voz
y_noise_looped = np.tile(y_noise, repeats)[:target_len]

# Mezclamos (bajamos el volumen del ruido a la mitad para que no ahogue la voz)
y_mix = y_speech + (0.5 * y_noise_looped)
print(f"   Mezcla final generada con loop. Duración: {len(y_mix)/SR_TARGET:.2f} s")

# ========================================================
# STFT Y NORMALIZACIÓN
# ========================================================
S_mix = librosa.stft(y_mix, n_fft=N_FFT, hop_length=HOP_LENGTH)
X_raw = np.abs(S_mix)

# Normalización para evitar NaN (Gradientes Explosivos)
X_real = X_raw / np.max(X_raw) 

print(f"\n2. Dimensiones de la Matriz X (Mezcla Normalizada): {X_real.shape}")

# ========================================================
# SPLIT CONTIGUO EN EL TIEMPO
# ========================================================
T = X_real.shape[1]
train_idx = int(0.70 * T)
val_idx = int(0.85 * T)

X_train = X_real[:, :train_idx]
X_val = X_real[:, train_idx:val_idx]
X_test = X_real[:, val_idx:]

print(f"   X_train (70%): {X_train.shape}")
print(f"   X_val   (15%): {X_val.shape}")
print(f"   X_test  (15%): {X_test.shape}\n")

# ========================================================
# ALGORITMO 2: EVALUACIÓN EN HELDOUT
# ========================================================
def eval_heldout(W_fixed, X_eval, innerH_eval=100, alpha_H=1e-3):
    m, k = W_fixed.shape
    n_eval = X_eval.shape[1]
    H_eval = np.random.rand(k, n_eval) / np.sqrt(k)
    
    for t in range(innerH_eval):
        g_H = W_fixed.T @ ((W_fixed @ H_eval) - X_eval)
        H_eval = np.maximum(H_eval - alpha_H * g_H, 0)
        
    rmse = np.sqrt(np.sum(((W_fixed @ H_eval) - X_eval)**2) / (m * n_eval))
    return rmse

# ========================================================
# PRUEBA DE RANGOS (k)
# ========================================================
k_values = [10, 20, 30, 40]
val_rmse_history = []
ITERATIONS = 400 

print(f"3. Evaluando diferentes valores de rango (k) con {ITERATIONS} iteraciones...")
for k_test in k_values:
    print(f"   Entrenando W con k={k_test} en el set de Train...")
    np.random.seed(42)
    
    W_train, _, _ = unified_bcgd(
        X=X_train, k=k_test, steps=ITERATIONS, 
        innerW=1, innerH=1, 
        alpha_W=1e-4, alpha_H=1e-4, 
        method='nesterov', beta=0.9, 
        is_nmf=True, lmbda=0.0
    )
    
    rmse_val = eval_heldout(W_train, X_val)
    val_rmse_history.append(rmse_val)
    print(f"   -> RMSE en Validación: {rmse_val:.4f}")

# Graficar Val RMSE vs k
plt.figure(figsize=(8, 5))
plt.plot(k_values, val_rmse_history, marker='o', color='#55A868', linewidth=2, markersize=8)
plt.title('Evaluación de Hiperparámetros: RMSE de Validación vs Rango (k)', fontsize=14, fontweight='bold')
plt.xlabel('Rango (k)', fontsize=12)
plt.ylabel('RMSE (Validación)', fontsize=12)
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# ========================================================
# EVALUACIÓN FINAL EN TEST
# ========================================================
best_k_idx = np.argmin(val_rmse_history)
best_k = k_values[best_k_idx]
print(f"\n4. El mejor k según validación fue k={best_k}. Evaluando en TEST...")

np.random.seed(42)
W_best, _, _ = unified_bcgd(X=X_train, k=best_k, steps=ITERATIONS, method='nesterov', is_nmf=True, alpha_W=1e-4, alpha_H=1e-4)

final_test_rmse = eval_heldout(W_best, X_test)
print(f"★ RMSE FINAL EN TEST: {final_test_rmse:.4f} ★")